# Graded Response Model — Single Scale

This notebook fits a standard **GRModel** (single latent dimension) to all 22
RWA items, treating the entire questionnaire as measuring one underlying
construct.

Two models are fit:
1. **Baseline** — ignorable missingness (missing responses have their log-likelihood zeroed out)
2. **Imputed** — missing responses are handled via **mixed imputation** (MICEBayesianLOO + baseline IRT), using analytic Rao-Blackwellization

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['TQDM_DISABLE'] = '1'
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from plot_helpers import (plot_loss_comparison, plot_forest_discriminations,
                          plot_ability_scatter, plot_ability_distributions,
                          plot_thresholds, plot_individual_abilities,
                          plot_imputation_weights_pcolormesh)

import pandas as pd
import polars as pl

from bayesianquilts.data.rwa import get_data, item_keys
from bayesianquilts.irt.grm import GRModel
from bayesianquilts.imputation.mice_loo import MICEBayesianLOO

## 1. Load the RWA Dataset

In [ ]:
response_cardinality = 9

df, num_people = get_data(polars_out=True)
print(f"Dataset: {num_people} people, {len(item_keys)} items, {response_cardinality} response categories (0-8)")
df.head()

In [ ]:
SUBSAMPLE_N = num_people
sub_df = df
print(f"Using full dataset: N = {SUBSAMPLE_N}")

## 2. Prepare Data and Fit the Baseline GRM

We use `GRModel` with `dim=1` so that all 22 items load on a single
latent trait. Missing/invalid responses (if any) have their
log-likelihood zeroed out (ignorable missingness).

In [ ]:
def make_data_dict(dataframe):
    """Convert polars DataFrame to dict of numpy float64 arrays."""
    data = {}
    for col in dataframe.columns:
        arr = dataframe[col].to_numpy().astype(np.float64)
        data[col] = arr
    data['person'] = np.arange(len(dataframe), dtype=np.float64)
    return data

batch = make_data_dict(sub_df)

# Check for missing/invalid values
n_bad = sum(
    np.sum(np.isnan(batch[k]) | (batch[k] < 0) | (batch[k] >= response_cardinality))
    for k in item_keys
)
print(f"Bad/missing values: {n_bad}")

BATCH_SIZE = 256
steps_per_epoch = int(np.ceil(SUBSAMPLE_N / BATCH_SIZE))
print(f"N: {SUBSAMPLE_N}, Batch size: {BATCH_SIZE}, Steps per epoch: {steps_per_epoch}")

def data_factory():
    indices = np.arange(SUBSAMPLE_N)
    np.random.shuffle(indices)
    for start in range(0, SUBSAMPLE_N, BATCH_SIZE):
        end = min(start + BATCH_SIZE, SUBSAMPLE_N)
        idx_batch = indices[start:end]
        yield {k: v[idx_batch] for k, v in batch.items()}

In [ ]:
model_baseline = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
)

NUM_EPOCHS = 200
SNAPSHOT_EPOCH = 50

res_baseline = model_baseline.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
    zero_nan_grads=True,
    snapshot_epoch=SNAPSHOT_EPOCH,
    lr_decay_factor=0.975,
)

losses_baseline = res_baseline[0]
snapshot_params = res_baseline[2] if len(res_baseline) > 2 else None
print(f"Baseline final loss: {losses_baseline[-1]:.2f}")
if snapshot_params is not None:
    print(f"Snapshot saved at epoch {SNAPSHOT_EPOCH}")

In [ ]:
model_baseline.save_to_disk('grm_baseline')
print("Baseline model saved to grm_baseline/")

In [ ]:
def calibrate_manually(model, n_samples=32, seed=42):
    try:
        surrogate = model.surrogate_distribution_generator(model.params)
        key = jax.random.PRNGKey(seed)
        samples = surrogate.sample(n_samples, seed=key)
        expectations = {k: jnp.mean(v, axis=0) for k, v in samples.items()}
        model.calibrated_expectations = expectations
        model.surrogate_sample = samples
    except KeyError as e:
        print(f"  Warning: surrogate sampling failed ({e}), using point estimates")
        point_estimates = {}
        for key_name, value in model.params.items():
            parts = key_name.split('\\')
            if len(parts) >= 4:
                param_name = parts[0]
                if parts[-2] == 'normal' and parts[-1] == 'loc':
                    point_estimates[param_name] = value
        model.calibrated_expectations = point_estimates

calibrate_manually(model_baseline, n_samples=32, seed=101)

## 3. Fit MICEBayesianLOO Imputation Model

We build pairwise univariate imputation models for each item pair
using **MICEBayesianLOO** with Pathfinder variational inference.
This allows us to predict a full categorical PMF over the 9 response
categories for any missing cell, conditioned on the observed items
in that row.

In [ ]:
# Convert to pandas and replace the -1 missing code with NaN
imputation_df = sub_df.select(item_keys).to_pandas()
imputation_df = imputation_df.replace(-1, float('nan'))

print(f"Imputation DataFrame shape: {imputation_df.shape}")
print(f"Missing values per item:")
print(imputation_df.isna().sum())

In [ ]:
mice_loo = MICEBayesianLOO(
    prior_scale=1.0,
    pathfinder_num_samples=100,
    pathfinder_maxiter=50,
    batch_size=512,
    verbose=True,
)

mice_loo.fit_loo_models(
    imputation_df,
    n_top_features=22,
    n_jobs=1,
    fit_zero_predictors=True,
)

print("\nMICE LOO model fitting complete.")

In [ ]:
mice_loo.save('mice_loo_model.yaml')
print("MICE LOO model saved to mice_loo_model.yaml")

In [ ]:
from bayesianquilts.imputation.mixed import IrtMixedImputationModel

mixed_imputation = IrtMixedImputationModel(
    irt_model=model_baseline,
    mice_model=mice_loo,
    data_factory=data_factory,
    irt_elpd_batch_size=4,
)

print(mixed_imputation.summary())

## 4. Fit GRM with Imputation

We now fit a second GRM that uses the trained `MICEBayesianLOO` model
to analytically marginalize over the imputation distribution for
missing cells (Rao-Blackwellization). Instead of discarding missing
observations, the model computes:

$$\text{contribution} = \log \sum_k q(k) \, p(Y=k \mid \phi)$$

where $q(k)$ is the imputation PMF and $p(Y=k \mid \phi)$ is the
GRM probability under the current parameters.

In [ ]:
model_imputed = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
    imputation_model=mixed_imputation,
)

if snapshot_params is not None:
    print(f"Warm-starting from baseline epoch-{SNAPSHOT_EPOCH} snapshot")

res_imputed = model_imputed.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=2e-4,
    patience=10,
    zero_nan_grads=True,
    initial_values=snapshot_params,
    lr_decay_factor=0.975,
)

losses_imputed = res_imputed[0]
print(f"Imputed final loss: {losses_imputed[-1]:.2f}")

In [ ]:
model_imputed.save_to_disk('grm_imputed')
print("Imputed model saved to grm_imputed/")

## 5. Training Diagnostics

In [ ]:
fig = plot_loss_comparison(losses_baseline, losses_imputed, title='RWA Scale')
fig.savefig('loss_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show()

## 6. Posterior Parameter Estimates

In [ ]:
# calibrate_manually already defined above; just calibrate the imputed model
calibrate_manually(model_imputed, n_samples=32, seed=102)

In [ ]:
fig = plot_forest_discriminations(item_keys, model_baseline, model_imputed, title='RWA — Item Discriminations')
fig.savefig('discriminations.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
ab_base = np.array(model_baseline.calibrated_expectations['abilities']).flatten()
ab_imp = np.array(model_imputed.calibrated_expectations['abilities']).flatten()

fig = plot_ability_scatter(ab_base, ab_imp, label='authoritarianism')
fig.savefig('ability_scatter.pdf', bbox_inches='tight', dpi=150)
plt.show()

fig = plot_ability_distributions(ab_base, ab_imp, label='authoritarianism')
fig.savefig('ability_distributions.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_thresholds(item_keys, model_baseline, model_imputed, title='RWA — Difficulty Thresholds')
fig.savefig('thresholds.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_individual_abilities(item_keys, model_baseline, model_imputed)
fig.savefig('individual_abilities.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_imputation_weights_pcolormesh(mice_loo, mixed_imputation, item_keys,
                                          title='RWA — Imputation Weights')
fig.savefig('imputation_weights.pdf', bbox_inches='tight', dpi=150)
plt.show()

## Summary

This notebook demonstrated two approaches to fitting a single-scale Graded
Response Model to all 22 RWA items using `GRModel` with `dim=1`:

1. **Baseline (ignorable missingness)** — missing responses are effectively
   skipped by zeroing out their log-likelihood contribution.

2. **Mixed imputation (MICE + baseline IRT)** — a `MICEBayesianLOO` pairwise
   Bayesian model is blended with the baseline IRT model's marginalized
   predictions via per-item softmax weights over ELPD scores:
   `PMF = w_mice × MICE + (1 − w_mice) × IRT_baseline`. The GRM then
   analytically marginalizes over the blended imputation distribution
   (Rao-Blackwellization), yielding zero-variance contributions for
   missing cells.

The side-by-side discrimination and ability plots above allow direct
comparison of the two strategies. When missingness is informative
(non-ignorable), the imputed model is expected to produce less biased
parameter estimates.